# 11주 · 등류·한계수심과 수면형

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 퀴즈 #3·GVF 방정식 / 통제단면·계산 방향

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week11/CE272_W11.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week11) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 각 S절 개별 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: 등류수심 이분법·수면형 비교

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

## S01. 원본 M2·S1 수면형 판별

**슬라이드 연결**: CE272_W11a_Steady_Nonuniform_Flow_3.pptx p7, 8, 9, 10, 16, 17, 18 / CE272_W11b_Steady_Nonuniform_Flow_4.pptx p21, 22, 23, 24, 25, 26, 27

**자료 구분**: 원본 비교 예제

**먼저 예측**: 같은 유량과 관측 수심이라도 수로 경사가 달라지면 수면형이 달라지는가?

**문제**: b=10 m, Q=50 m³/s, n=0.02인 직사각형 수로의 한계수심과 S0=0.001·0.005 각각의 등류수심을 구하시오. 관측 수심 1.50 m의 수면형을 판별하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
b,Q,n,y=10.,50.,.02,1.5
yc=hyd.critical_depth(Q,b)
rows=[]
for S0 in [.001,.005]:
    yn=hyd.normal_depth(Q,b,0,n,S0)
    Sf=hyd.friction_slope(Q,y,b,0,n)
    Fr=hyd.froude(Q,y,b)
    rows.append({'S0':S0,'yn_m':yn,'yc_m':yc,'observed_y_m':y,'Fr':Fr,
                 'dy_dx_downstream':(S0-Sf)/(1-Fr**2),
                 'Manning_flow_residual':hyd.manning(yn,b,0,n,S0)-Q})
pd.DataFrame(rows)

**검토 문제**: yn>y>yc와 y>yc>yn의 차이를 M2·S1에 연결하시오. 한계수심 근처에서 GVF 식을 그대로 적분하기 어려운 이유를 설명하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## S02. 저수지 입구의 통제와 S2

**슬라이드 연결**: CE272_W11b_Steady_Nonuniform_Flow_4.pptx p29, 30, 31, 32, 33

**자료 구분**: 원본 저수지 유입 예제

**먼저 예측**: 저수지 수두 6 m를 입구 수심 6 m로 바로 둘 수 있는가?

**문제**: 저수지 접근유속과 입구손실을 무시한다. b=10 m, S0=0.010, n=0.013인 수로에서 저수지 수두 H0=6 m, 입구 한계류를 가정하여 유량을 구하시오. 한계경사와 등류수심으로 가정을 확인하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
b,S0,n,H0=10.,.010,.013,6.
yc=2*H0/3
Q=b*np.sqrt(hyd.G*yc**3)
s=hyd.section(yc,b)
Sc=(n*Q/s['A'])**2/s['R']**(4/3)
yn=hyd.normal_depth(Q,b,0,n,S0)
pd.Series({'critical_depth_m':yc,'Q_m3_s':Q,'critical_slope':Sc,
           'bed_slope':S0,'normal_depth_m':yn,
           'entrance_energy_check_m':hyd.specific_energy(Q,yc,b)})

**검토 문제**: 입구 한계류와 하류 등류수심 사이의 S2 수면형을 설명하시오. 입구손실을 추가하면 먼저 수정할 방정식을 제시하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## 학습 기록

수심의 대소·Fr·마찰경사로 수면형과 계산 방향을 설명하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록

In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')